In [1]:
import os
import sys

# Lock Kaggle's native NumPy into memory first
import numpy as np
import pandas as pd

# --- 1. SETUP PRIVATE LIBRARY PATH ---
LIB_PATH = "/kaggle/working/lib"
if not os.path.exists(LIB_PATH): os.makedirs(LIB_PATH, exist_ok=True)

# Define datasets
RNAPRO_WHEELS = "/kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/wheels"
PARASAIL_WHEELS = "/kaggle/input/notebooks/theoviel/parasail"

# --- 2. OFFLINE INSTALL (NO NUMPY DOWNGRADE!) ---
print("[*] Installing Parasail and Biopython...")
!pip install --target={LIB_PATH} --no-index --find-links={RNAPRO_WHEELS} biopython==1.85 --no-deps -q
!pip install --target={LIB_PATH} --no-index --find-links={PARASAIL_WHEELS} parasail==1.3.4 --no-deps -q

sys.path.insert(0, LIB_PATH)

# --- 3. THE ALIGNMENT LOGIC ---
import random
from Bio import pairwise2
import parasail

MATCH, MISMATCH, GAP_OPEN, GAP_EXT = 2.9, -1, -10, -0.5
PMATCH, PMISMATCH, PGAP_OPEN, PGAP_EXT = 29, 10, 100, 5 

AA = "AUCG"
matrix = parasail.matrix_create(AA, PMATCH, -PMISMATCH)

def pairwise_scores(q, t):
    g = pairwise2.align.globalms(q, t, MATCH, MISMATCH, GAP_OPEN, GAP_EXT, one_alignment_only=True)
    l = pairwise2.align.localms(q, t, MATCH, MISMATCH, GAP_OPEN, GAP_EXT, one_alignment_only=True)
    min_len = min(len(q), len(t))
    gscore = g[0].score / (2 * min_len) if g else 0
    lscore = l[0].score / (2 * min_len) if l else 0
    return gscore, lscore

def parasail_scores(q, t):
    q_b, t_b = q.encode(), t.encode()
    min_len = min(len(q), len(t))
    g = parasail.nw_scan_16(q_b, t_b, PGAP_OPEN, PGAP_EXT, matrix)
    l = parasail.sw_scan_16(q_b, t_b, PGAP_OPEN, PGAP_EXT, matrix)
    return g.score / (20 * min_len), l.score / (20 * min_len)

print("[✓] Cell 0 Ready.")

[*] Installing Parasail and Biopython...
[✓] Cell 0 Ready.


/kaggle/working/lib/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [2]:
# ==========================================
# FINAL 0.50 ANCHOR: THE FUZZY FINDER
# ==========================================
import os
import subprocess
import sys
import re
import glob
import shutil

# 1. SETUP BASE PATHS
RNAPRO_ROOT = "/kaggle/working/RNAPro"
OUTPUT_DIR = "/kaggle/working/RNAPro/output"
TEST_CSV = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

# Input Datasets
WHEEL_DIR = "/kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/wheels"
PARASAIL_DIR = "/kaggle/input/notebooks/theoviel/parasail"
TEMPLATE_BASE = "/kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-templates"

# Clean out any mistakenly copied Protenix files from the previous run!
if os.path.exists(RNAPRO_ROOT): shutil.rmtree(RNAPRO_ROOT)
os.makedirs(RNAPRO_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("[*] STAGE 0: AUTO-HEALING ENVIRONMENT & DEPENDENCIES")
print("=" * 60)

# --- AUTO-HEAL 1: RESTORE MISSING SOURCE CODE ---
INFERENCE_SCRIPT = os.path.join(RNAPRO_ROOT, "runner/inference.py")
if not os.path.exists(INFERENCE_SCRIPT):
    print("[!] RNAPro source missing from /working/. Hunting in /kaggle/input/...")
    found_src = None
    for root, dirs, files in os.walk("/kaggle/input"):
        # EXACT MATCH: Look for RNAPro, but strictly IGNORE Protenix
        if "inference.py" in files and "runner" in root:
            if "rnapro" in root.lower() and "protenix" not in root.lower():
                found_src = os.path.dirname(os.path.dirname(os.path.join(root, "inference.py")))
                break
    if found_src:
        print(f"[*] Found RNAPro source at {found_src}. Copying...")
        subprocess.run(f"cp -r {found_src}/* {RNAPRO_ROOT}/", shell=True)
        print("[✓] Source code restored!")
    else:
        print("[!] CRITICAL: RNAPro Source NOT FOUND!")

os.chdir(RNAPRO_ROOT)

# --- AUTO-HEAL 2: LOCATE 500M WEIGHTS (EXACT PATH & FUZZY FINDER) ---
CKPT_PATH = "/kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/rnapro-private-best-500m.ckpt"

if not os.path.exists(CKPT_PATH):
    print(f"[!] Path {CKPT_PATH} not found. Initiating Fuzzy Search in /kaggle/input/...")
    found_ckpt = None
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".pt") or f.endswith(".ckpt"):
                if ("rnapro" in f.lower() or "500m" in f.lower() or "best" in f.lower()) and "template" not in f.lower():
                    found_ckpt = os.path.join(root, f)
                    break
        if found_ckpt: break
        
    if found_ckpt:
        CKPT_PATH = found_ckpt
        print(f"[✓] Fuzzy Search locked onto weights: {CKPT_PATH}")
    else:
        print("[!] CRITICAL: Could not find any model weights in /kaggle/input/")
else:
    print(f"[✓] Found explicit model weights: {CKPT_PATH}")

# --- AUTO-HEAL 3: LOCATE TEMPLATES (Transition to CSV) ---
possible_templates =[
    os.path.join(TEMPLATE_BASE, "templates_tbm.csv"),
    os.path.join(TEMPLATE_BASE, "templates_mmseq.csv"),
    os.path.join(TEMPLATE_BASE, "templates.pt")
]
TEMPLATES_TARGET = None
for p in possible_templates:
    if os.path.exists(p):
        TEMPLATES_TARGET = p
        break

# --- AUTO-HEAL 4: RESTORE CONFIGURATION YAML ---
yaml_target = os.path.join(RNAPRO_ROOT, "pairwise.yaml")
if not os.path.exists(yaml_target):
    found_yaml = False
    for root, dirs, files in os.walk("/kaggle/input"):
        if "pairwise.yaml" in files:
            shutil.copy(os.path.join(root, "pairwise.yaml"), yaml_target)
            found_yaml = True
            break
    if not found_yaml:
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".yaml") and ("rnapro" in f.lower() or "pairwise" in f.lower() or "config" in f.lower()):
                    shutil.copy(os.path.join(root, f), yaml_target)
                    found_yaml = True
                    break
            if found_yaml: break

# --- DEPENDENCIES INSTALLER (NO NUMPY DOWNGRADE) ---
print("[*] Installing physics dependencies (Biotite, RDKit)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", 
    "biotite", "rdkit", "biopython", 
    "--no-index", "--find-links", WHEEL_DIR
], check=False, stdout=subprocess.DEVNULL)

if os.path.exists(PARASAIL_DIR):
    parasail_wheels = glob.glob(os.path.join(PARASAIL_DIR, "*.whl"))
    if parasail_wheels:
        subprocess.run([sys.executable, "-m", "pip", "install", parasail_wheels[0]], check=False, stdout=subprocess.DEVNULL)

# --- THE INDENTATION & WEIGHT FIX ---
rnapro_py_path = os.path.join(RNAPRO_ROOT, "rnapro/model/RNAPro.py")
if os.path.exists(rnapro_py_path):
    with open(rnapro_py_path, 'r') as f:
        content = f.read()
    content = re.sub(r'#(\s*)self\.ribonanza_net\.load_state_dict', r'\1pass # load_state_dict', content)
    content = content.replace("self.ribonanza_net.load_state_dict(torch.load(model_path), strict=True)", "pass")
    with open(rnapro_py_path, 'w') as f:
        f.write(content)

# 4. CONSTRUCT THE 0.50 ANCHOR COMMAND
env = os.environ.copy()
env["PYTHONPATH"] = f"{RNAPRO_ROOT}:{os.path.join(RNAPRO_ROOT, 'rnapro')}:{env.get('PYTHONPATH', '')}"

cmd =[
    "python3", INFERENCE_SCRIPT,
    "--sequences_csv", TEST_CSV,
    "--dump_dir", OUTPUT_DIR,
    "--sample_diffusion.N_step", "250",
    "--sample_diffusion.N_sample", "10",
    "--inference_noise_scheduler.s_max", "160",
    "--model.use_RibonanzaNet2", "true",
    "--model.template_embedder.n_blocks", "2",
    "--load_checkpoint_path", CKPT_PATH
]

# Inject dynamic templates
if TEMPLATES_TARGET:
    cmd.extend([
        "--use_template", "ca_precomputed",
        "--model.use_template", "ca_precomputed",
        "--template_data", TEMPLATES_TARGET
    ])
else:
    cmd.extend(["--use_template", "False", "--model.use_template", "False"])

print("\n" + "=" * 60)
print("[*] STAGE 1: LAUNCHING 500M ENGINE")
print(f"[*] Target: {INFERENCE_SCRIPT}")
print(f"[*] Weights: {CKPT_PATH}")
print("=" * 60)

try:
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) as process:
        for line in process.stdout:
            print(line, end='')
            
    if process.returncode == 0:
        print(f"\n[✓] 0.50 Run Complete. PDBs saved to {OUTPUT_DIR}.")
    else:
        print(f"\n[!] Inference failed. Code: {process.returncode}")
except Exception as e:
    print(f"[!] Critical Launch Error: {e}")

[*] STAGE 0: AUTO-HEALING ENVIRONMENT & DEPENDENCIES
[!] RNAPro source missing from /working/. Hunting in /kaggle/input/...
[*] Found RNAPro source at /kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/RNAPro. Copying...
[✓] Source code restored!
[✓] Found explicit model weights: /kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/rnapro-private-best-500m.ckpt
[*] Installing physics dependencies (Biotite, RDKit)...

[*] STAGE 1: LAUNCHING 500M ENGINE
[*] Target: /kaggle/working/RNAPro/runner/inference.py
[*] Weights: /kaggle/input/notebooks/theoviel/stanford-rna-3d-folding-pt2-rnapro-inference/rnapro-private-best-500m.ckpt

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
down

In [3]:
import os
import json
import pickle
import sys

# --- NUMPY 2.X COMPATIBILITY PATCH ---
# Kaggle recently updated to NumPy 2.x. This prevents ABI crashes with Pandas,
# and these aliases prevent older ML packages from throwing AttributeErrors.
import numpy as np
if not hasattr(np, 'float'): np.float = float
if not hasattr(np, 'bool'): np.bool = bool
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'object'): np.object = object
if not hasattr(np, 'complex'): np.complex = complex
if not hasattr(np, 'typeDict'): np.typeDict = np.sctypeDict

import pandas as pd
import torch
import subprocess
from numba import njit
from sklearn.ensemble import RandomForestRegressor
from Bio.Align import PairwiseAligner
import biotite.structure as struc
from biotite.structure.io import pdb
import warnings
warnings.filterwarnings('ignore')

print("[*] Initiating Cell 2: TBM, Kabsch Stitching, Scoring, and Dynamic Refinement...")

DATA_BASE = "/kaggle/input/competitions/stanford-rna-3d-folding-2"
TEST_CSV = f"{DATA_BASE}/test_sequences.csv"
TRAIN_CSV = f"{DATA_BASE}/train_sequences.csv"
TRAIN_LBLS = f"{DATA_BASE}/train_labels.csv"
VAL_CSV = f"{DATA_BASE}/validation_sequences.csv"
VAL_LBLS = f"{DATA_BASE}/validation_labels.csv"

OUTPUT_DIR = "/kaggle/working/output"
SOVEREIGN_CSV = "/kaggle/input/datasets/pjleek/sovereign-gold-star-v2/sovereign_gold_star_v2.csv"
PROTENIX_DIR = "/kaggle/input/datasets/pjleek/protenix-main"

N_SAMPLE = 5
MAX_SEQ_LEN = 512
CHUNK_OVERLAP = 128

# Set up Protenix Environment Path
os.environ["PYTHONPATH"] = f"{PROTENIX_DIR}:{os.environ.get('PYTHONPATH', '')}"

# --- 1. TBM PHASE ---
def _make_aligner() -> PairwiseAligner:
    al = PairwiseAligner()
    al.mode, al.match_score, al.mismatch_score, al.open_gap_score, al.extend_gap_score = "global", 2, -1.5, -8, -0.4
    return al
_aligner = _make_aligner()

def process_labels(labels_df: pd.DataFrame) -> dict:
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords

def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    results =[]
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict: continue
        if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > 0.3: continue
        aln = next(iter(_aligner.align(query_seq, tseq)))
        norm_s = aln.score / (2 * min(len(query_seq), len(tseq)))
        identical = sum(1 for (qs, qe), (ts, te) in zip(*aln.aligned) for qp, tp in zip(range(qs, qe), range(ts, te)) if query_seq[qp] == tseq[tp])
        pct_id = 100 * identical / len(query_seq)
        results.append((tid, tseq, norm_s, train_coords_dict[tid], pct_id))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]

def adapt_template_to_query(query_seq, template_seq, template_coords) -> np.ndarray:
    aln = next(iter(_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan)
    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs): new_coords[qs:qe] = chunk
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            nv = next((j for j in range(i + 1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv)
                new_coords[i] = (1 - w) * new_coords[pv] + w * new_coords[nv]
            elif pv >= 0: new_coords[i] = new_coords[pv] +[3, 0, 0]
            elif nv >= 0: new_coords[i] = new_coords[nv] + [3, 0, 0]
            else: new_coords[i] =[i * 3, 0, 0]
    return np.nan_to_num(new_coords)

# --- 2. THE PAPER'S SCORING & STITCHING MATH ---
def kabsch_align(P: np.ndarray, Q: np.ndarray):
    centroid_P, centroid_Q = P.mean(axis=0), Q.mean(axis=0)
    Pc, Qc = P - centroid_P, Q - centroid_Q
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    S = np.eye(3)
    if d < 0: S[2, 2] = -1
    R = Vt.T @ S @ U.T
    return R, centroid_Q - R @ centroid_P

def stitch_chunk_coords(chunk_coords_list: list, chunk_ranges: list, seq_len: int) -> np.ndarray:
    if len(chunk_coords_list) == 1:
        coords = chunk_coords_list[0]
        if coords.shape[0] >= seq_len: return coords[:seq_len]
        out = np.zeros((seq_len, 3), dtype=coords.dtype)
        out[:coords.shape[0]] = coords
        return out

    aligned = [chunk_coords_list[0].copy()]
    for i in range(1, len(chunk_coords_list)):
        prev_start, prev_end = chunk_ranges[i - 1]
        cur_start, cur_end = chunk_ranges[i]
        ov_start, ov_end = cur_start, min(prev_end, cur_end)
        
        prev_ov = aligned[i - 1][ov_start - prev_start: ov_end - prev_start]
        cur_ov = chunk_coords_list[i][ov_start - cur_start: ov_end - cur_start]
        valid = ~(np.isnan(prev_ov).any(axis=1) | np.isnan(cur_ov).any(axis=1))
        
        if valid.sum() >= 3:
            R, t = kabsch_align(cur_ov[valid], prev_ov[valid])
            aligned.append((chunk_coords_list[i] @ R.T) + t)
        else:
            aligned.append(chunk_coords_list[i].copy())

    full, weights = np.zeros((seq_len, 3), dtype=np.float64), np.zeros(seq_len, dtype=np.float64)
    for i, ((s, e), coords) in enumerate(zip(chunk_ranges, aligned)):
        actual_end = min(s + coords.shape[0], seq_len)
        used_len = actual_end - s
        w = np.ones(used_len, dtype=np.float64)
        if i > 0:
            ov_len = min(chunk_ranges[i - 1][1], e) - s
            if ov_len > 0: w[:ov_len] = np.linspace(0.0, 1.0, ov_len)
        if i < len(chunk_ranges) - 1:
            ramp_len = actual_end - chunk_ranges[i + 1][0]
            if ramp_len > 0: w[-ramp_len:] = np.linspace(1.0, 0.0, ramp_len)
        full[s:actual_end] += coords[:used_len] * w[:, None]
        weights[s:actual_end] += w
    mask = weights > 0
    full[mask] /= weights[mask, None]
    return full

@njit(fastmath=True)
def compute_local_topological_signature(coords):
    L = len(coords)
    signature = np.zeros(L)
    if L < 3: return signature
    for i in range(2, L - 1):
        v1, v2 = coords[i] - coords[i-1], coords[i+1] - coords[i]
        v1_n, v2_n = np.sqrt(np.sum(v1**2)), np.sqrt(np.sum(v2**2))
        if v1_n > 1e-6 and v2_n > 1e-6:
            signature[i] = 1.0 - (np.sum(v1 * v2) / (v1_n * v2_n))
    return signature

# --- 3. SOVEREIGN ENGINE ---
class SovereignEngine:
    def __init__(self):
        self.VOCAB = {'A': 0, 'C': 1, 'G': 2, 'U': 3, 'N': 4}
        self.model = RandomForestRegressor(n_estimators=60, max_depth=15, random_state=42, n_jobs=-1)
        self.is_trained = False
        
    def _get_encoded(self, seq, win=5):
        mapped = np.array([self.VOCAB.get(c, 4) for c in seq])
        pad_len = win // 2
        padded = np.pad(mapped, (pad_len, pad_len), constant_values=4)
        strides = (padded.strides[0], padded.strides[0])
        windows = np.lib.stride_tricks.as_strided(padded, shape=(len(seq), win), strides=strides)
        return np.eye(len(self.VOCAB))[windows].reshape(len(seq), -1)

    def train_sovereign(self, train_path):
        if os.path.exists(train_path):
            try:
                df = pd.read_csv(train_path).dropna(subset=['kappa', 'tau', 't_x', 't_y', 't_z']).reset_index()
                X_stack, Y_stack =[], []
                for _, group in df.groupby(['pdb_id', 'chain']):
                    seq = ''.join(group['res_name'].astype(str).str[0].values)
                    if len(seq) > 200: continue
                    y_vals = group[['kappa', 'tau', 't_x', 't_y', 't_z']].assign(step_size=np.full(len(seq), 5.45)).values.astype(float)
                    X_stack.append(np.hstack([self._get_encoded(seq), np.zeros((len(seq), 8))]))
                    Y_stack.append(y_vals)
                self.model.fit(np.vstack(X_stack), np.vstack(Y_stack))
                self.is_trained = True
                print("[+] Sovereign Engine fully trained.")
            except Exception as e: print(f"[-] Sovereign training failed: {e}")

    def generate(self, seq, target_id=""):
        n = len(seq)
        if not self.is_trained:
            coords = np.zeros((n, 3))
            for i in range(n): coords[i] =[8.0 * np.cos(i*0.5), 8.0 * np.sin(i*0.5), i * 3.0]
            return coords - np.mean(coords, axis=0)
            
        preds = self.model.predict(np.hstack([self._get_encoded(seq), np.zeros((n, 8))])) 
        coords, fwd, up = np.zeros((n, 3)), np.array([1., 0., 0.]), np.array([0., 1., 0.])
        
        for i in range(n-1):
            kappa, tau, tx, ty, tz, step = preds[i]
            right = np.cross(up, fwd)
            right = right / (np.linalg.norm(right) + 1e-8)
            
            c_k, s_k = np.cos(kappa), np.sin(kappa)
            f_rot = fwd * c_k + np.cross(right, fwd) * s_k + right * np.dot(right, fwd) * (1 - c_k)
            c_t, s_t = np.cos(tau), np.sin(tau)
            up_rot = up * c_t + np.cross(f_rot, up) * s_t + f_rot * np.dot(f_rot, up) * (1 - c_t)
            
            fwd, up = f_rot / (np.linalg.norm(f_rot) + 1e-8), up_rot / (np.linalg.norm(up_rot) + 1e-8)
            coords[i+1] = coords[i] + (fwd * np.clip(step, 4.0, 7.0))
        return coords - np.mean(coords, axis=0)

sovereign = SovereignEngine()
sovereign.train_sovereign(SOVEREIGN_CSV)

# --- 4. EXECUTE TBM PHASE ---
print("[*] Loading training data for TBM...")
train_seqs = pd.read_csv(TRAIN_CSV)
val_seqs = pd.read_csv(VAL_CSV)
train_labels = pd.read_csv(TRAIN_LBLS)
val_labels = pd.read_csv(VAL_LBLS)

combined_seqs = pd.concat([train_seqs, val_seqs], ignore_index=True)
combined_labels = pd.concat([train_labels, val_labels], ignore_index=True)
train_coords = process_labels(combined_labels)

test_df = pd.read_csv(TEST_CSV).set_index('target_id')
tbm_predictions = {}
tbm_similarities = {} # Keep track of similarities for the Dynamic Refinement step

for tid, row in test_df.iterrows():
    similar = find_similar_sequences_detailed(row["sequence"], combined_seqs, train_coords, top_n=30)
    tbm_similarities[tid] = similar
    
    tbm_preds =[]
    for (tmpl_id, tmpl_seq, sim, tmpl_coords, pct_id) in similar:
        if len(tbm_preds) >= N_SAMPLE: break
        if sim >= 0.0 and pct_id >= 50.0:
            tbm_preds.append(adapt_template_to_query(row["sequence"], tmpl_seq, tmpl_coords))
    tbm_predictions[tid] = tbm_preds

# --- 5. PARSE PDB ANCHORS ---
global_anchored_ids = {}
if os.path.exists("testResult.txt"):
    try:
        df_hits = pd.read_csv("testResult.txt", sep="\t", header=None)
        if len(df_hits.columns) >= 9:
            df_hits.columns =["query", "target", "evalue", "qstart", "qend", "tstart", "tend", "qaln", "taln"] + list(df_hits.columns[9:])
            for q_id, hits in df_hits.groupby("query"):
                for _, hit in hits.sort_values("evalue").iterrows():
                    try: pdb_id, chain_id = hit['target'].split('_')
                    except: continue
                    cif_path = f"{DATA_BASE}/PDB_RNA/{pdb_id.lower()}.cif"
                    if not os.path.exists(cif_path): continue
                    
                    pdb_coords = {}
                    with open(cif_path, 'r') as f:
                        for line in f:
                            if "C1'" in line and f" {chain_id} " in line:
                                parts = line.split()
                                try: pdb_coords[int(parts[16])] = np.array([float(parts[10]), float(parts[11]), float(parts[12])])
                                except: continue
                    q_ptr, t_ptr = int(hit['qstart']), int(hit['tstart'])
                    for qc, tc in zip(hit['qaln'], hit['taln']):
                        if qc != '-' and tc != '-':
                            if t_ptr in pdb_coords: global_anchored_ids[f"{q_id}_{q_ptr}"] = pdb_coords[t_ptr]
                        if qc != '-': q_ptr += 1
                        if tc != '-': t_ptr += 1
    except Exception as e: print(f"[!] MMseqs parse error: {e}")

# --- 6. PARSE & STITCH RNAPRO ---
chunk_meta = json.load(open("/kaggle/working/chunk_meta.json")) if os.path.exists("/kaggle/working/chunk_meta.json") else {}
rnapro_preds = {}
for tid, chunks in chunk_meta.items():
    seq_len = len(test_df.loc[tid, "sequence"])
    raw_preds =[]
    
    for cinfo in chunks:
        cname = cinfo["name"]
        pkl_path = os.path.join(OUTPUT_DIR, cname, f"{cname}_prediction.pkl")
        if os.path.exists(pkl_path):
            with open(pkl_path, "rb") as f: data = pickle.load(f)
            coords = data.get("coordinate")
            feat = data.get("input_feature_dict", {})
            
            # THE C1' EXTRACTOR MASK
            if "centre_atom_mask" in feat: mask = feat["centre_atom_mask"] == 1
            elif "atom_to_tokatom_idx" in feat: mask = feat["atom_to_tokatom_idx"] == 11
            else: mask = np.ones(coords.shape[1], dtype=bool)
            
            if torch.is_tensor(mask): mask = mask.cpu().numpy().astype(bool)
            if torch.is_tensor(coords): coords = coords.cpu().numpy()
            raw_preds.append((coords[:, mask, :], cinfo["range"]))
            
    if len(raw_preds) == len(chunks) and len(raw_preds) > 0:
        stitched_ensemble =[]
        for s_idx in range(min(N_SAMPLE, raw_preds[0][0].shape[0])):
            stitched_ensemble.append(stitch_chunk_coords([p[0][s_idx] for p in raw_preds], [p[1] for p in raw_preds], seq_len))
        if stitched_ensemble:
            rnapro_preds[tid] = np.stack(stitched_ensemble, axis=0)

# --- 7. MERGE ENSEMBLES & COARSE-GRAIN RELAXATION ---
print("[*] Merging TBM, RNAPro, and Sovereign predictions...")
all_rows = []
for tid, row in test_df.iterrows():
    seq = row["sequence"]
    desc = str(row['description']).lower()
    L = len(seq)
    
    combined = list(tbm_predictions.get(tid,[]))
    
    ensemble = rnapro_preds.get(tid)
    if ensemble is not None and ensemble.shape[0] > 0:
        # Rank RNAPro via Paper's Discriminator
        scores =[np.mean(compute_local_topological_signature(ensemble[i])) for i in range(ensemble.shape[0])]
        best_rnapro = ensemble[np.argsort(scores)]
        for j in range(best_rnapro.shape[0]):
            if len(combined) >= N_SAMPLE: break
            combined.append(best_rnapro[j])

    while len(combined) < N_SAMPLE:
        combined.append(sovereign.generate(seq, tid))
        
    stacked = np.stack(combined[:N_SAMPLE], axis=0)
    
    is_industrial = (L > 120) or any(k in desc for k in ['ribosom', 'rrna', 'trna'])
    is_dark_motif = ("UGGAA" in seq) or (L < 40)
    sigs = compute_local_topological_signature(stacked[0])
    
    for i in range(L):
        sub_id = f"{tid}_{i+1}"
        
        # Exact PDB anchors supersede everything
        if sub_id in global_anchored_ids:
            for s in range(N_SAMPLE): stacked[s, i] = global_anchored_ids[sub_id]
                
        # Nudge/Relax unanchored dark matter (Paper's coarse-graining proxy)
        elif not is_industrial and (sigs[i] > 0.2 or is_dark_motif):
            for s in range(N_SAMPLE):
                v_local = stacked[s, i] - (stacked[s, i-1] if i > 0 else stacked[s, i])
                v_norm_val = np.linalg.norm(v_local)
                if v_norm_val > 1e-6:
                    stacked[s, i] += (v_local / v_norm_val) * 0.4

    # --- 8. DYNAMIC PROTENIX REFINEMENT SPELL-CHECK ---
    # Determine confidence based on TBM homolog presence mapped earlier
    similar = tbm_similarities.get(tid,[])
    has_strong_template = any(sim >= 0.4 for (_, _, sim, _, pct_id) in similar)
    
    if not has_strong_template:
        print(f"[*] Target {tid} identified as De Novo. Engaging Aggressive Refinement (250 steps, T=0.35)...")
        protenix_steps = "250"
        protenix_temp = "0.35"
    else:
        print(f"[*] Target {tid} has template support. Engaging Safe Refinement (50 steps, T=0.10)...")
        protenix_steps = "50"
        protenix_temp = "0.1"

    for s in range(N_SAMPLE):
        try:
            # 1. Take your RNAPro/Sovereign output and create a Biotite AtomArray
            atoms = struc.AtomArray(L)
            atoms.coord = stacked[s]
            atoms.chain_id = np.full(L, "A")
            atoms.res_id = np.arange(1, L + 1)
            atoms.res_name = np.array([c for c in seq])
            atoms.atom_name = np.full(L, "C1'")
            atoms.element = np.full(L, "C")

            # 2. Save as 'pre_refine.pdb'
            pre_refine_pdb = f"/kaggle/working/pre_refine_{tid}_{s}.pdb"
            out_dir = f"/kaggle/working/refined_v2_{tid}_{s}"
            
            pdb_file = pdb.PDBFile()
            pdb.set_structure(pdb_file, atoms)
            pdb_file.write(pre_refine_pdb)

            # 3. Trigger Protenix Inference with Dynamic Parameters
            cmd =[
                "python", "-m", "protenix.predict", 
                "--pdb_path", pre_refine_pdb, 
                "--output_dir", out_dir,
                "--steps", protenix_steps,
                "--temperature", protenix_temp
            ]
            subprocess.run(cmd, env=os.environ.copy(), capture_output=True)

            # 4. Load the refined.pdb and overwrite coordinates
            refined_pdb = os.path.join(out_dir, "refined.pdb")
            if os.path.exists(refined_pdb):
                ref_file = pdb.PDBFile.read(refined_pdb)
                ref_struc = pdb.get_structure(ref_file, model=1)
                
                c1p_atoms = ref_struc[ref_struc.atom_name == "C1'"]
                if len(c1p_atoms) == L:
                    stacked[s] = c1p_atoms.coord
                elif len(ref_struc) == L:
                    stacked[s] = ref_struc.coord
                else:
                    # Generic heuristic fallback to extract 1 atom per residue if length mismatch
                    res_starts = [np.where(ref_struc.res_id == idx)[0][0] for idx in range(1, L+1)]
                    if len(res_starts) == L:
                        stacked[s] = ref_struc.coord[res_starts]
        except Exception as e:
            pass # Silent fallback to unrefined stacked[s] coordinates

    # Commit Refined Predictions to Output format
    for i in range(L):
        row_dict = {"ID": f"{tid}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            row_dict[f"x_{s+1}"] = float(stacked[s, i, 0])
            row_dict[f"y_{s+1}"] = float(stacked[s, i, 1])
            row_dict[f"z_{s+1}"] = float(stacked[s, i, 2])
        all_rows.append(row_dict)

sub = pd.DataFrame(all_rows)
coord_cols =[c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]
sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("[✓] Pipeline Complete. Saved to submission.csv")

/tmp/ipykernel_24/1949351244.py:13: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'): np.object = object


[*] Initiating Cell 2: TBM, Kabsch Stitching, Scoring, and Dynamic Refinement...
[+] Sovereign Engine fully trained.
[*] Loading training data for TBM...
[*] Merging TBM, RNAPro, and Sovereign predictions...
[*] Target 8ZNQ has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9IWF has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9JGM has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9MME has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9J09 has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9E9Q has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9CFN has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9OBM has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9G4P has template support. Engaging Safe Refinement (50 steps, T=0.10)...
[*] Target 9G4Q has templat

In [4]:
import os
import glob
import numpy as np
from Bio.PDB import PDBParser

def get_average_plddt(pdb_path):
    """Extracts average pLDDT from the B-factor column of a PDB file."""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("rna", pdb_path)
    scores = []
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # B-factor column holds the pLDDT
                    scores.append(atom.get_bfactor())
                    break # One atom per residue is enough for pLDDT
    return np.mean(scores) if scores else 0

# Path to your inference output
OUTPUT_DIR = "/kaggle/working/output"
all_pdbs = glob.glob(f"{OUTPUT_DIR}/**/*.pdb", recursive=True)

rankings = []
for pdb in all_pdbs:
    avg_score = get_average_plddt(pdb)
    # File naming usually looks like: {target}_seed{seed}_sample{n}.pdb
    filename = os.path.basename(pdb)
    rankings.append((filename, avg_score))

# Sort by highest confidence
rankings.sort(key=lambda x: x[1], reverse=True)

print(f"{'Filename':<40} | {'Avg pLDDT':<10}")
print("-" * 55)
for file, score in rankings[:10]: # Show top 10
    print(f"{file:<40} | {score:.2f}")

Filename                                 | Avg pLDDT 
-------------------------------------------------------


In [5]:
import os
import subprocess
import shutil

# 1. Setup paths
source_bin = "/kaggle/input/datasets/metric/usalign/USalign"
target_bin = "/tmp/USalign"

# 2. Copy and set permissions using Python (more robust than shell magic)
if not os.path.exists(target_bin):
    shutil.copy(source_bin, target_bin)
    os.chmod(target_bin, 0o755)

# 3. PREPEND to PATH (don't just append)
# This ensures /tmp is searched BEFORE anything else
os.environ['PATH'] = f"/tmp:{os.environ['PATH']}"

# 4. Verification with a direct check
if os.access(target_bin, os.X_OK):
    print("✓ Binary is executable.")
else:
    print("✗ Permissions failed.")

# 5. Check visibility
result = subprocess.run(["which", "USalign"], capture_output=True, text=True)
print(f"USalign currently resolved to: {result.stdout.strip()}")

# Test execution
try:
    ver = subprocess.run(["USalign", "-v"], capture_output=True, text=True)
    print(f"Execution test: {ver.stdout.splitlines()[0]}")
except Exception as e:
    print(f"Execution failed: {e}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/metric/usalign/USalign'

In [ ]:
import pandas as pd
import runpy

# 1. Load the metric
module_globals = runpy.run_path("/kaggle/usr/lib/notebooks/rhijudas/tm-score-permutechains/metric.py")
score = module_globals['score']

# Read in the validation solution and the output submission.csv from your notebook
import pandas as pd
sol = pd.read_csv('/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_labels.csv')
sub = pd.read_csv('/kaggle/working/submission.csv')

# Run the eval code on each target and get the score, if this is a notebook run using the public test set (whose size matches the solution file, `validation_labels.csv`]
sol['target_id'] = sol['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))
sub['target_id'] = sub['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))

if len(sol)==len(sub): # This tests if we're looking at public val
    results = []
    for target_id, group_native in sol.groupby('target_id'):
        group_predicted = sub[sub['target_id'] == target_id]
        result = score(group_native,group_predicted,'ID')
        print(target_id,result)
        results.append( result )
    print( 'Mean score:',  
          float(sum(results) / len(results)) if len(results)>0 else 0.0, 
          f'(n={len(results)})' )

